**Machine Learning**

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedKFold

In [4]:
train_eng = pd.read_parquet('data/clean/train_eng.parquet')

print(f"Loaded train_eng shape: {train_eng.shape}")

cat_count = len(train_eng.select_dtypes(include=['category']).columns)
print(f"Categorical columns strictly preserved: {cat_count}")

Loaded train_eng shape: (307511, 86)
Categorical columns strictly preserved: 16


In [5]:
y = train_eng['TARGET']
X = train_eng.drop(columns=['TARGET'])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
roc_aucs = []
pr_aucs = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.05,
        objective='binary',
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=0) 
        ]
    )

    y_pred_proba = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = y_pred_proba

    fold_roc = roc_auc_score(y_val, y_pred_proba)
    fold_pr = average_precision_score(y_val, y_pred_proba) 
    
    roc_aucs.append(fold_roc)
    pr_aucs.append(fold_pr)

    print(f"Fold {fold} | ROC AUC: {fold_roc:.4f} | PR AUC: {fold_pr:.4f}")

print("-" * 30)
print(f"Mean ROC AUC: {np.mean(roc_aucs):.4f} (±{np.std(roc_aucs):.4f})")
print(f"Mean PR AUC:  {np.mean(pr_aucs):.4f} (±{np.std(pr_aucs):.4f})")

[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.025743 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7022
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 81
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432482
[LightGBM] [Info] Start training from score -2.432482
Fold 1 | ROC AUC: 0.7608 | PR AUC: 0.2444
[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.021135 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7080
[LightGBM] [Info] Number of data points in the train set: 246009, 